# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² mental health survey dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Get metadata object
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

All entities, including record sets, fields, and columns, are referenced by their `@id`. Below we inspect available record sets and field ids.

In [ ]:
# List all record sets and their @ids
record_sets = metadata.recordSet

if not record_sets:
    print("No record sets listed in metadata. Attempting to infer available record sets via dataset.records().")
    # Try to get all record sets by inspecting dataset.records()
    try:
        # mlcroissant supports .list_record_sets()
        record_set_ids = dataset.list_record_sets()
        print("Record Set @ids:")
        for rset_id in record_set_ids:
            print(f"- {rset_id}")
    except Exception as e:
        print("Could not automatically enumerate record sets.")
else:
    record_set_ids = [rset['@id'] if isinstance(rset, dict) and '@id' in rset else rset for rset in record_sets]
    print("Record Set @ids from metadata:")
    for rset_id in record_set_ids:
        print(f"- {rset_id}")

# For this dataset, we expect at least one main record set, let's enumerate its fields by record set @id.
# We'll try the first if available
example_record_set = None
if record_set_ids:
    example_record_set = record_set_ids[0]
    print(f"\nInspecting fields for record set: {example_record_set}")
    try:
        # Get a single record to show the fields
        for x in dataset.records(record_set=example_record_set):
            print(f"Sample record keys: {list(x.keys())}")
            break
    except Exception as e:
        print("Could not get sample records for record set.")
else:
    print("No record set to inspect.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use record set and field `@id`s as referenced above.

In [ ]:
# Extract data from each record set
# If record_set_ids not defined, repeat previous step.
if 'record_set_ids' not in locals():
    try:
        record_set_ids = dataset.list_record_sets()
    except Exception:
        record_set_ids = []

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
    else:
        df = pd.DataFrame()
    dataframes[record_set_id] = df

# Display available columns for the main record set
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"Columns for record set {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    print(dataframes[main_record_set_id].head())
else:
    print("No data available for extraction.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.
For demonstration, let's analyze the GAD-7 numerical assessment column using its `@id` as found above. (If fields are named, ensure you use the exact field/column `@id`; update accordingly.)

In [ ]:
# Select a numeric field for analysis
main_record_set = list(dataframes.keys())[0] if dataframes else None
df = dataframes[main_record_set] if main_record_set else pd.DataFrame()

# Let's look for likely numeric fields; here, let's suppose GAD-7 total score is named by @id 'gad7_total_score'
numeric_field_id = 'gad7_total_score'  # Update to exact @id from earlier overview

# If not found, search for numeric columns
if numeric_field_id not in df.columns:
    numeric_columns = df.select_dtypes(include='number').columns.tolist()
    if numeric_columns:
        numeric_field_id = numeric_columns[0]
        print(f"Numeric field selected: {numeric_field_id}")
    else:
        print("No numeric fields found.")

threshold = 10
if numeric_field_id in df.columns:
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group field exploration (example, group by 'gender' if available; update to @id)
    group_field_id = 'gender'  # Update to relevant @id from previous section
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())
else:
    print(f"Field {numeric_field_id} not found in dataframe columns.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below, visualize the distribution of the GAD-7 score (or other available numeric field) and group comparisons.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of the numeric field
if numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id], bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id} scores")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If group field present, plot groupwise distribution
    if group_field_id in df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} scores by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print(f"Cannot plot; numeric field {numeric_field_id} not found.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² Kilifi Mental Health Survey dataset enables structured analysis of demographic and symptom scores, referenced reliably using `@id`.
- Using `mlcroissant` simplifies loading and processing Croissant-packaged datasets.
- Example analyses (filtering, normalization, grouping) and visualizations help highlight population trends relevant to mental health interventions.

**Further steps:**
- Explore relationships between scores and other demographic indicators using column `@id` references.
- Apply ML or statistical tests for deeper insights.